# ⚡ 05: Laya Non-Autoregressive Calibrated Decision & Probability Evaluation

**Enterprise Multi-Modal Financial RAG Benchmark**  
**Document:** JPMorgan Chase & Co. 2025 Annual Report  
**Scope:** 20 Easy Benchmark Questions (`GT-E-001` through `GT-E-020`)  
**Evaluator Architecture:** [Laya](https://github.com/NandhaKishorM/laya) (System 1 Decision Engine, Non-Autoregressive, Calibrated Probabilities)

---

### Why Laya for RAG Evaluation?
Traditional LLM-as-a-Judge requires calling an autoregressive model (e.g. 26B parameters) that generates text token-by-token over 5 to 15 seconds per question. In contrast, **Laya**:
1. **Single Forward Pass (~33 ms):** Non-autoregressive decision making with ~100x to 200x latency reduction.
2. **Calibrated Probabilities:** Trained with reinforcement learning against strictly proper scoring rules (RLCD).
   - `"type": "noul"` returns a true calibrated probability $P(\text{Yes}) \in [0.0, 1.0]$.
3. **Typed Decisions:**
   - **Typed Choice:** Categorical classification over discrete criteria.
   - **Typed Score:** Discrete ordinal ranking (e.g. `["inaccurate", "partially accurate", "fully accurate"]`).
   - **Calibrated Binary:** Exact probability of factual correctness, hallucination, or criteria presence.


In [1]:
import sys
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == "evaluation":
    BASE_DIR = BASE_DIR.parent
sys.path.insert(0, str(BASE_DIR))

from laya import Router
from agent.tools import retrieve_chunks, generate_grounded_answer

# Initialize Laya Router
router = Router()
print("✅ Laya System 1 Decision Router initialized.")


[*] Agent Orchestrator  : Ollama 'qwen3.5:9b' @ http://localhost:11434
✅ Laya System 1 Decision Router initialized.


## 1. Load Ground Truth Dataset (20 Easy Questions)


In [2]:
gt_path = BASE_DIR / "evaluation" / "ground_truth_dataset.json"
with open(gt_path, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

easy_questions = [q for q in gt_data["questions"] if q.get("difficulty") == "easy"]
print(f"Loaded {len(easy_questions)} Easy benchmark questions.")


Loaded 20 Easy benchmark questions.


## 2. Define Calibrated Probability Evaluator via Laya
We format each evaluation as a composite state:
```
[Question]: ...
[Candidate Answer]: ...
[Ground Truth]: ...
[Retrieved Context]: ...
```
And define questions with calibrated types:
- `is_accurate` (`noul`): Probability candidate matches ground truth.
- `is_grounded` (`noul`): Probability candidate is fully grounded in retrieved context without hallucination.
- `quality_tier` (`score`): Ordinal quality rating across `["poor", "acceptable", "flawless"]`.
- `failure_mode` (`choice`): Classification of any defect (`["none", "numeric_error", "hallucination", "incomplete"]`).


In [3]:
def evaluate_with_laya(query: str, gen_ans: str, gt_ans: str, context: str, criteria: list) -> dict:
    """Evaluates a RAG generation using Laya's non-autoregressive decision engine."""
    state = (
        f"QUESTION: {query}\n"
        f"CANDIDATE ANSWER: {gen_ans}\n"
        f"GROUND TRUTH: {gt_ans}\n"
        f"RETRIEVED CONTEXT: {context[:2000]}\n"
        f"CRITERIA: {', '.join(criteria)}"
    )
    
    questions = {
        "is_accurate": {
            "type": "noul",
            "instructions": "Does the candidate answer convey the correct facts and numbers as specified in the ground truth?"
        },
        "is_grounded": {
            "type": "noul",
            "instructions": "Is the candidate answer completely supported by the retrieved context without hallucinating external or false data?"
        },
        "quality_tier": {
            "type": "score",
            "instructions": "What is the overall financial response quality?",
            "criteria": ["inaccurate", "acceptable", "flawless"]
        },
        "defect_type": {
            "type": "choice",
            "instructions": "If there is any flaw in the candidate answer, categorize it.",
            "criteria": {
                "none": "The answer is accurate and well-grounded",
                "numeric_mismatch": "The numbers, dollars, or percentages disagree with ground truth",
                "hallucination": "Information present in answer but missing from retrieved context",
                "omission": "Answer is incomplete or missed key required criteria"
            }
        }
    }
    
    t0 = time.perf_counter()
    pred = router.predict(state, questions)
    latency_ms = (time.perf_counter() - t0) * 1000.0
    
    answers = pred.get("answers", {})
    return {
        "p_accurate": float(answers.get("is_accurate", {}).get("noul", 0.0)),
        "p_grounded": float(answers.get("is_grounded", {}).get("noul", 0.0)),
        "quality_score": answers.get("quality_tier", {}).get("score", "acceptable"),
        "defect_category": answers.get("defect_type", {}).get("choice", "none"),
        "model_used": pred.get("routing", {}).get("model", "unknown"),
        "latency_ms": latency_ms
    }

print("✅ Laya decision evaluation pipeline configured.")


✅ Laya decision evaluation pipeline configured.


## 3. Run Laya Evaluation Across 20 Easy Questions


In [4]:
laya_records = []

for idx, q in enumerate(easy_questions, 1):
    qid = q["id"]
    query = q["question"]
    gt_ans = q["ground_truth_answer"]
    criteria = q.get("evaluation_criteria", [])
    
    # 1. Retrieve
    context = retrieve_chunks.invoke({"query": query, "limit": 5})
    
    # 2. Synthesize with Groq (Llama 3.3 70B)
    gen_ans = generate_grounded_answer.invoke({
        "question": query,
        "retrieved_evidence": context,
        "provider": "groq",
    })
    
    # 3. Laya Non-Autoregressive Evaluation
    laya_res = evaluate_with_laya(query, gen_ans, gt_ans, context, criteria)
    
    record = {
        "id": qid,
        "question": query,
        "category": q.get("category"),
        "modality": q.get("source_modality"),
        "p_accurate": laya_res["p_accurate"],
        "p_grounded": laya_res["p_grounded"],
        "quality_score": laya_res["quality_score"],
        "defect_category": laya_res["defect_category"],
        "latency_ms": laya_res["latency_ms"],
        "generated_answer": gen_ans,
        "ground_truth_answer": gt_ans
    }
    laya_records.append(record)
    print(f"[{idx:02d}/20] {qid}: P(Accurate)={record['p_accurate']:.3f} | P(Grounded)={record['p_grounded']:.3f} | Quality={record['quality_score']} ({record['latency_ms']:.1f}ms)")

laya_df = pd.DataFrame(laya_records)
print("\n✅ Completed Laya evaluation for all 20 questions.")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\user\Documents\Gen_AI\Multi-modal-RAG\.venv\Lib\site-packages\laya\router.py:456: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, **kwargs)


[01/20] GT-E-001: P(Accurate)=0.801 | P(Grounded)=0.825 | Quality=1.5507 (3847.6ms)
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[02/20] GT-E-002: P(Accurate)=0.792 | P(Grounded)=0.764 | Quality=1.2363 (1987.9ms)
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[03/20] GT-E-003: P(Accurate)=0.822 | P(Grounded)=0.842 | Quality=1.4866 (2087.9ms)
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[04/20] GT-E-004: P(Accurate)=0.820 | P(Grounded)=0.797 | Quality=1.3145 (268.2ms)
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[05/20] GT-E-005: P(Accurate)=0.800 | P(Grounded)=0.801 | Quality=1.4462 (2219.6ms)
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[06/20] GT-E-006: P(Accurate)=0.751 | P(Grounded)=0.768 | Quality=1.3826 (1357.2ms)
[*] Answer Synthesizer  : Groq 'openai/gpt-oss-20b' (explicit)
[07/20] GT-E-007: P(Accurate)=0.865 | P(Grounded)=0.865 | Quality=1.544 (2272.5ms)
[*] Answer Synthesizer  : Groq 'open

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\user\Documents\Gen_AI\Multi-modal-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\user\Documents\Gen_AI\Multi-modal-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `hug

OSError: [WinError 1314] A required privilege is not held by the client: '..\\..\\..\\blobs\\00e35f88bb731bb9126a914666ab1cdac8a204c8' -> 'C:\\Users\\user\\.cache\\huggingface\\hub\\models--convaiinnovations--laya\\snapshots\\55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851\\multilingual\\rl_agent_config.json'

## 4. Laya Calibrated Summary Scorecard


In [ ]:
laya_summary = pd.DataFrame([
    {"Laya Metric": "Mean Calibrated P(Accurate)", "Value": f"{laya_df['p_accurate'].mean():.3f} ({laya_df['p_accurate'].mean()*100:.1f}%)"},
    {"Laya Metric": "Mean Calibrated P(Grounded)", "Value": f"{laya_df['p_grounded'].mean():.3f} ({laya_df['p_grounded'].mean()*100:.1f}%)"},
    {"Laya Metric": "High Confidence Accuracy (P > 0.80)", "Value": f"{(laya_df['p_accurate'] >= 0.80).sum()} / {len(laya_df)}"},
    {"Laya Metric": "High Confidence Groundedness (P > 0.80)", "Value": f"{(laya_df['p_grounded'] >= 0.80).sum()} / {len(laya_df)}"},
    {"Laya Metric": "Average Decision Latency", "Value": f"{laya_df['latency_ms'].mean():.1f} ms"},
    {"Laya Metric": "Dominant Defect Category", "Value": f"{laya_df['defect_category'].mode()[0]}"}
])
laya_summary


## 5. Visualizing Calibrated Probabilities & Defect Analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Laya Non-Autoregressive Evaluation Profile (20 Easy Questions)", fontsize=16, fontweight="bold", y=1.02)

# Chart 1: Calibrated Probability Scatter (Accuracy vs Groundedness)
axes[0].scatter(laya_df["p_grounded"], laya_df["p_accurate"], c="#2563eb", s=80, edgecolors="black", alpha=0.8)
axes[0].set_title("Calibrated P(Grounded) vs P(Accurate)", fontweight="semibold")
axes[0].set_xlabel("P(Grounded)")
axes[0].set_ylabel("P(Accurate)")
axes[0].set_xlim(-0.05, 1.05)
axes[0].set_ylim(-0.05, 1.05)
axes[0].axvline(0.8, color="green", linestyle="--", alpha=0.5, label="High Confidence (0.8)")
axes[0].axhline(0.8, color="green", linestyle="--", alpha=0.5)
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# Chart 2: Quality Tier Distribution
quality_counts = laya_df["quality_score"].value_counts()
axes[1].bar(quality_counts.index, quality_counts.values, color=["#10b981", "#3b82f6", "#ef4444"], edgecolor="black", alpha=0.85)
axes[1].set_title("Quality Tier Breakdown", fontweight="semibold")
axes[1].set_ylabel("Question Count")
axes[1].grid(axis="y", linestyle="--", alpha=0.6)

# Chart 3: Evaluation Latency Distribution
axes[2].hist(laya_df["latency_ms"], bins=6, color="#f59e0b", edgecolor="black", alpha=0.85)
axes[2].set_title("Laya Forward-Pass Latency (ms)", fontweight="semibold")
axes[2].set_xlabel("Latency (ms)")
axes[2].set_ylabel("Question Count")
axes[2].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


## 6. Throughput Benchmark: Laya vs. Autoregressive LLM-as-a-Judge


In [ ]:
avg_laya_ms = laya_df["latency_ms"].mean()
# Typical 26B Gemma / 9B Qwen eval time on local Ollama is ~7,500 ms per question
typical_llm_ms = 7500.0
speedup = typical_llm_ms / max(avg_laya_ms, 1.0)

print(f"⚡ Average Laya decision latency: {avg_laya_ms:.1f} ms")
print(f"🐢 Typical 26B Autoregressive LLM-as-a-Judge: ~{typical_llm_ms:.0f} ms")
print(f"🚀 Speedup Factor: ~{speedup:.1f}x faster evaluation!")
